In [ ]:
from math import ceil
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sb
import pickle as pkl
from infer import NOVICModel, utils
import itertools
from PIL import Image
from matplotlib import cm
import torch

result_folder = ""

def float01_to_uint8_image(array, im_min=None, im_max=None):
    array = np.asarray(array)
    if array.ndim == 3 and array.shape[0] in (1, 3) and array.shape[-1] not in (1, 3):
        array = np.transpose(array, (1, 2, 0))
    # print(array.min(), array.max())
    if im_min is None:
      im_min = array.min()
    if im_max is None:    im_max = array.max()
    array -= im_min
    array /= im_max - im_min
    array = np.clip(array, 0.0, 1.0)
    return (array * 255).round().astype(np.uint8)

def show(img, im_min=None, im_max=None, **kwargs):
  img = np.array(img)
  if img.shape[0] == 3:
    img = img.transpose(1, 2, 0)
  if im_min is None:
    im_min = img.min()
  if im_max is None:    im_max = img.max()
  img -= im_min
  img /= im_max - im_min

  # img -= img.min();img /= img.max()

  plt.imshow(img, **kwargs); plt.axis('off')
  return img

def mask_show(img, **kwargs):
  img = np.array(img)
  if img.shape[0] == 3:
    img = img.transpose(1, 2, 0)

  img -= img.min()
  plt.imshow(img, **kwargs); plt.axis('off')

def show_bias_crops(concept_parameters, label, concept, amount=5):
  crops_u = concept_parameters[label]["crops_u"]
  crops = concept_parameters[label]["crops"]
  if "crop_masks" in concept_parameters[label]:
    masks = concept_parameters[label]["crop_masks"]
  best_crops_ids = np.argsort(crops_u[:, concept])[::-1][:amount]
  best_crops = crops[best_crops_ids]
  im_min, im_max = crops.min(), crops.max()
  if "crop_masks" in concept_parameters[label]:
    best_masks = masks[best_crops_ids]
  imgs = []
  crop_w, crop_h = 3, 3  # constant size per crop (inches)
  plt.figure(figsize=(crop_w * amount, crop_h))
  for i in range(amount):
    plt.subplot(1, amount, i + 1)
    imgs.append(show(best_crops[i], im_min=im_min, im_max=im_max))
  plt.tight_layout()
  plt.show()

  if "crop_masks" in concept_parameters[label]:
    plt.figure(figsize=(crop_w * amount, crop_h))
    for i in range(amount):
      plt.subplot(1, amount, i + 1)
      mask_show(best_masks[i])
    plt.tight_layout()
    plt.show()
  print('\n\n')
  return imgs

def softmax(x):
    return np.exp(x)/sum(np.exp(x))

utils.allow_tf32(enable=True)
novicmodel = NOVICModel(checkpoint='outputs/ovod_20240628_142131/ovod_chunk0433_20240630_235415.train')

In [ ]:
def get_best_labels(model, concept_parameters, label, concept, amount=100):
    crops_u = concept_parameters[label]["crops_u"]
    crops = concept_parameters[label]["crops"]
    best_crops_ids = np.argsort(crops_u[:, concept])[::-1][:amount]
    best_crops = crops[best_crops_ids]
    im_min, im_max = crops.min(), crops.max()
    images = [Image.fromarray(float01_to_uint8_image(crop, im_min=im_min, im_max=im_max)) for crop in best_crops[:amount]]
    res = {}
    with model:
        output = model.classify_image(image=images)
        # print('TOP-3 PREDICTIONS:', ' / '.join(f'{pred} = {prob * 100:.3g}%' for pred, prob in itertools.islice(zip(output.preds[0], output.probs[0]), 3)))
        best_crops_u = crops_u[:, concept][best_crops_ids[:amount]]
        for i in range(len(output.preds)):
            mult = best_crops_u[i] / best_crops_u.sum()
            # print(output.probs[i][:3])
            probas = softmax(0.5 * np.array(output.probs[i][:3]))
            for j, prob in enumerate(probas):
                pred = output.preds[i][j]
                if pred in res:
                    res[pred] += prob * mult
                else:
                    res[pred] = prob * mult
    res = [(key, value) for (key, value) in res.items()]
    res.sort(key=lambda x: x[1], reverse=True)
    return res

In [ ]:
exp_name = "CMNISTb"
concept_id = 8
patch_id = 6
exp_type = "b"

label_results = {}
for exp_id in range(10):
    with open(f"{result_folder}/models/{exp_name}/model_{exp_id}.pkl", "rb") as f:
        model = pkl.load(f)
    with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        concept_res = pkl.load(f)
    with open(f"{result_folder}/models/{exp_name}/debiasing_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
        debias_res = pkl.load(f)
    for el in debias_res["bias_estimator"]["bias_vectors"]:
        bias_names = get_best_labels(novicmodel, concept_res["concept_parameters"], label=el[0], concept=el[1], amount=100)
        total_value = sum([value for (_, value) in bias_names[:10]])
        if el[0] not in label_results:
            label_results[el[0]] = {}
        for name, value in bias_names[:10]:
            if name in label_results[el[0]]:
                label_results[el[0]][name] += value / total_value
            else:
                label_results[el[0]][name] = value / total_value
for label in label_results:
    label_results[label] = [(key, value) for (key, value) in label_results[label].items()]
    label_results[label].sort(key=lambda x: x[1], reverse=True)
    print(f"Most predictive labels for label {label}:")
    for name, value in label_results[label][:10]:
        print(f"{name}: {value:.4f}")
    print(f"Top 10: {', '.join([name for (name, _) in label_results[label][:10]])}")

In [ ]:
exp_name = "CMNISTb"
concept_id = 8
patch_id = 6
exp_id = 0
exp_type = "b"
with open(f"{result_folder}/models/{exp_name}/model_{exp_id}.pkl", "rb") as f:
    model = pkl.load(f)
with open(f"{result_folder}/models/{exp_name}/concepts_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
    concept_res = pkl.load(f)
with open(f"{result_folder}/models/{exp_name}/debiasing_{exp_id}_{exp_type}_{concept_id}_{patch_id}.pkl", "rb") as f:
    debias_res = pkl.load(f)
print(f"Base accuracy: {model['correctness_matrix'].sum()/model['appearance_matrix'].sum()}, Debiased accuracy: {debias_res['debiasing_impact'][0].sum()/debias_res['debiasing_impact'][1].sum()}")
fig, axes = plt.subplots(1, 2, figsize=(15,5))
sb.heatmap(model["correctness_matrix"]/model["appearance_matrix"], annot=True, ax=axes[0], vmin=0, vmax=1)
axes[0].set_title("Correctness matrix")
sb.heatmap(debias_res["debiasing_impact"][0]/debias_res["debiasing_impact"][1] - model["correctness_matrix"]/model["appearance_matrix"], annot=True, ax=axes[1], center=0, vmin=-0.1, vmax=0.1)
axes[1].set_title("Debiasing impact")
plt.show()
for el in debias_res["bias_estimator"]["bias_vectors"]:
    print(f"Label {el[0]}, concept {el[1]}, chi2 stat: {debias_res['chi2_test']['merged'][el[0]]['chi2_stats'][debias_res['cluster_labels'][concept_id * el[0] + el[1]]]}, p-value: {debias_res['chi2_test']['merged'][el[0]]['p_values'][debias_res['cluster_labels'][concept_id * el[0] + el[1]]]}, mcc value: {debias_res['chi2_test']['merged'][el[0]]['mcc_values'][debias_res['cluster_labels'][concept_id * el[0] + el[1]]]}, bias estimation: {debias_res['bias_estimator']['all_values'][el[0]][el[1]]}")
    bias_names = get_best_labels(novicmodel, concept_res["concept_parameters"], label=el[0], concept=el[1], amount=100)
    print(f"Most predictive labels for this concept: {', '.join([name for (name, _) in bias_names[:10]])}")
    show_bias_crops(concept_parameters=concept_res["concept_parameters"], label=el[0], concept=el[1], amount=8)